# Primary Odor Descriptor
## FRAGRANCE INGREDIENT GLOSSARY
## https://d3t14p1xronwr0.cloudfront.net/docs/ifra-fragrance-ingredient-glossary-april-2020.pdf

In [1]:
import random
from typing import List
import pandas as pd
import numpy as np

IN_CSV  = "Data/FIG_with_isomeric_smiles.csv"
ALL_OUT = "primary_odor_descriptor_all.csv"
SUB_OUT = "primary_odor_descriptor_175.csv"
SEED    = 42
N_SUB   = 175

random.seed(SEED)
np.random.seed(SEED)

REQUIRED_COLS = [
    "CAS number",
    "Principal name",
    "PUBCHEM_ISOMERIC_SMILES",
    "Primary descriptor",
    "Descriptor 2",
    "Descriptor 3",
]

def clean_str(x):
    return "" if pd.isna(x) else str(x).strip()

def build_options(answer: str, d2: str, d3: str, uniques: List[str]) -> str:
    ans = clean_str(answer)
    ex2 = clean_str(d2)
    ex3 = clean_str(d3)

    pool = [u for u in uniques if u and u != ans and u != ex2 and u != ex3]
    random.shuffle(pool)
    distractors = pool[:3]

    if len(distractors) < 3:
        fallback = [u for u in uniques if u and u != ans and u not in distractors]
        random.shuffle(fallback)
        distractors += fallback[: 3 - len(distractors)]

    opts = [ans] + distractors[:3]
    random.shuffle(opts)
    return ";".join(opts)

def stratified_sample_equal(df: pd.DataFrame, label_col: str, n: int, seed: int) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    groups = df.groupby(label_col, dropna=False)
    labels = list(groups.groups.keys())
    k = len(labels)
    if k == 0:
        return df.head(0)

    base = max(n // k, 0)
    base_alloc = {lab: min(base, len(groups.get_group(lab))) for lab in labels}
    allocated = sum(base_alloc.values())
    remaining = max(n - allocated, 0)

    capacities = [(lab, len(groups.get_group(lab)) - base_alloc[lab]) for lab in labels]
    capacities = [(lab, cap) for (lab, cap) in capacities if cap > 0]

    rng.shuffle(labels)
    while remaining > 0 and capacities:
        for i, (lab, cap) in enumerate(list(capacities)):
            if remaining <= 0:
                break
            if cap > 0:
                base_alloc[lab] += 1
                remaining -= 1
                capacities[i] = (lab, cap - 1)
        capacities = [(lab, cap) for (lab, cap) in capacities if cap > 0]

    parts = []
    for lab, take in base_alloc.items():
        if take <= 0:
            continue
        g = groups.get_group(lab)
        parts.append(g.sample(n=min(take, len(g)), random_state=seed))
    out = pd.concat(parts, ignore_index=True) if parts else df.head(0)

    if len(out) > n:
        out = out.sample(n=n, random_state=seed).reset_index(drop=True)
    elif len(out) < n:
        remaining_pool = df.loc[~df.index.isin(out.index)]
        if len(remaining_pool) > 0:
            extra = remaining_pool.sample(n=min(n - len(out), len(remaining_pool)), random_state=seed)
            out = pd.concat([out, extra], ignore_index=True)
        out = out.sample(frac=1.0, random_state=seed).reset_index(drop=True)
        out = out.head(n)
    return out

def main():
    df = pd.read_csv(IN_CSV)

    # Check columns
    missing = [c for c in REQUIRED_COLS if c not in df.columns]
    if missing:
        raise SystemExit(f"❌ Missing required columns: {missing}")

    # --- drop of empty/NaN SMILES ---
    before = len(df)
    # Normalize to string, strip, and convert common empties to NaN
    smi = df["PUBCHEM_ISOMERIC_SMILES"].astype(str).str.strip()
    empties = {"", "nan", "none", "null", "na", "n/a", "nil"}
    smi = smi.where(~smi.str.lower().isin(empties))
    df["PUBCHEM_ISOMERIC_SMILES"] = smi
    df = df.dropna(subset=["PUBCHEM_ISOMERIC_SMILES"]).copy()
    after = len(df)
    print(f"Filtered rows by SMILES: kept {after}/{before} (dropped {before - after})")

    # Unique labels for options
    df["Primary descriptor"] = df["Primary descriptor"].astype(str).str.strip()
    unique_labels = sorted({u for u in df["Primary descriptor"].unique() if u})

    # Build rows
    rows = []
    for _, r in df.iterrows():
        pname = clean_str(r["Principal name"])
        smi   = clean_str(r["PUBCHEM_ISOMERIC_SMILES"])
        pdesc = clean_str(r["Primary descriptor"])
        d2    = clean_str(r.get("Descriptor 2", ""))
        d3    = clean_str(r.get("Descriptor 3", ""))

        options = build_options(pdesc, d2, d3, unique_labels)

        rows.append({
            "compound.name_1": pname,
            "SMILES_1": smi,
            "compound.name_2": "",
            "SMILES_2": "",
            "OPTIONS": options,
            "question_category": "primary_odor_descriptor",
            "prompt.1": f"What is the primary odor descriptor of molecule [{smi}]? Select only one from the [OPTIONS]. Do not write any comments.",
            "prompt.2": f"What is the primary odor descriptor of molecule [{pname}]? Select only one from the [OPTIONS]. Do not write any comments.",
            "answer": pdesc,
            "other_info": f"Descriptor 2={d2};Descriptor 3={d3}",
        })

    out_all = pd.DataFrame(rows)
    out_all.insert(0, "question_ID", np.arange(1, len(out_all) + 1, dtype=int))
    out_all = out_all.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    out_all["question_ID"] = np.arange(1, len(out_all) + 1, dtype=int)

    out_all.to_csv(ALL_OUT, index=False)
    print(f"Wrote {ALL_OUT} (n={len(out_all)})")

    # Stratified 175-row subset by Primary descriptor (the 'answer' column)
    subset = stratified_sample_equal(out_all, label_col="answer", n=N_SUB, seed=SEED)
    subset = subset.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    subset["question_ID"] = np.arange(1, len(subset) + 1, dtype=int)

    subset.to_csv(SUB_OUT, index=False)
    print(f"Wrote {SUB_OUT} (n={len(subset)})")
    print("Label counts in subset:")
    print(subset["answer"].value_counts().sort_index())

if __name__ == "__main__":
    main()


Filtered rows by SMILES: kept 2175/3119 (dropped 944)
Wrote primary_odor_descriptor_all.csv (n=2175)
Wrote primary_odor_descriptor_175.csv (n=175)
Label counts in subset:
answer
Acidic           7
Aldehydic        7
Amber            7
Animal like      7
Anisic           7
Aromatic         7
Balsamic         7
Camphoraceous    4
Citrus           7
Earthy           7
Floral           7
Food Like        7
Fruity           7
Gourmand         7
Green            7
Herbal           7
Honey            3
Marine           7
Minty            7
Musk like        7
Ozonic           6
Powdery          6
Smoky            6
Spicy            6
Sulfurous        6
Tobacco like     6
Woody            6
Name: count, dtype: int64
